# Gradient steering — Latent DiT · z=16 · window 4

**The question this architecture was built to answer.** In 128-d observation space, frozen-probe gradients are
adversarial-dominated on every architecture tried (GRU, MSE transformer, pixel DiT): the readout flips, the
content doesn't move, cos(δ, Δ_true) ≈ 0.1–0.25. One candidate explanation is **dimensionality of adversarial
freedom** — a 128-d surface has a huge subspace of readout-changing directions orthogonal to anything semantic.
A **16-d VAE code** has almost none, and the frozen decoder is a second, unconditional manifold projector.
Does steering in the compressed space fix controllability, or does the failure survive compression (which would
locate it in belief dynamics rather than representation geometry)?

**Data / model provenance.** `0_latent_dit_z16_w4` = **Latent DiT · z=16 · window 4** (frozen `vae_z16`
recon RMSE 0.1293 vs noisy / 0.0984 vs clean; d256 4-layer DiT core over 16-d latents), checkpoint
`runs/latent_dit/0_latent_dit_z16_w4/best_model.pt`; rows copied from `../latent_DiT/LATENT_DIT_RUNS.md`.
Quality gate + probe grid: `../latent_DiT/latent_dit_world_state.ipynb`. Dataset
`datasets/4_fixed_refl_inview`, edit frame `ef`=20, N=64 edits, K=15 rollout. §4 metrics from
`scripts/editability_metrics.py`; probes from `pim.extractors.fit_readability_probes`.

## Definitions — the three write surfaces

This architecture has **two** clean write surfaces where the pixel DiT had one, plus the diffusion iterate:

| editor | surface it perturbs | mechanism |
|---|---|---|
| **Obs Grad Steering (n, λ)** | the **observation** frames fed as history (128-d each) | Adam (300 steps, lr 0.02) on δ over the newest n observation frames, backprop through the **VAE encoder** and the core; loss `‖probe(state(encode(obs+δ))) − target‖² + λ‖δ‖²`, obs clamped to [0,1]. The direct analogue of the pixel notebooks' "Input Grad" |
| **Latent-window Grad Steering (n, λ)** | the carried **latent** window (16-d per frame) | Adam (300 steps, lr 0.02) on δ over the newest n latents of the state buffer — the compressed surface; **this is where the "no room for adversarial directions" hypothesis lives** |
| **Iterate Grad Steering @(Lℓ, τ)** | the **diffusion iterate** mid-generation | pause the fresh-noise Euler ODE at τ_pause, Adam on the iterate until a probe fit **and held-out-verified at that same (ℓ, τ)** reads the target, resume the remaining steps ("pause–optimize–resume") |

Each runs through both a frozen **linear** and a frozen **MLP** (standard 2×256, `STD_EPOCHS`=300) probe.

| reference / metric | definition | units | better |
|---|---|---|---|
| **Render write @1 (oracle)** | newest latent ← `encode(clean render of the edited world)`; single-frame oracle content | — | — |
| **Counterfactual window write (n=W, oracle)** | **all W** window latents ← encodes of clean renders of the velocity-consistent counterfactual world (edited object keeps its own velocity on a path offset by a constant Δ so it lands exactly on the target at `ef`). On the pixel DiT this reached Edit Index **+0.71** — the architecture's editability ceiling | — | — |
| **Δ_true** | the true edit direction, measured **in the space the editor writes in**: obs-space `gt_edited − obs[ef−1]`; latent-space `encode(gt_edited) − encode(obs[ef−1])` | — | — |
| **cos(δ, Δ_true) / angle** | per-sample cosine and angle; chance from an empirical shuffled-pair control. **The headline diagnostic**: in latent space the hypothesis predicts a much higher cosine than the ≈0.1–0.25 seen in observation space | — / ° | ↑ / ↓ |
| **probe residual** | `‖probe(state) − target‖` after steering (did the optimization succeed at all) | sim units | ↓ |
| **Edit Index / zone RMSEs / GT-traj RMSE / fidelity ratio** | canonical §4 set (`../METRICS_AND_EDITORS.md` §4); Edit Index ∈ [−1, +1], +1 = the edited world, −1 = the unedited world, ≈0 = equidistant **or garbage** | — | ↑ (index) |

Rollouts are mean-mode free-runs with the feedback loop kept **in latent space** (no VAE round-trip per step);
step 0 decodes frame `ef`. Observation-space errors are scored against the **clean** render throughout.

In [ ]:
# [1] Setup: model, dataset, probes on the activations state (linear + MLP, held-out verified).
import os, sys
sys.path.insert(0, "../../../..")
sys.path.insert(0, "../../../../scripts")
import numpy as np, torch, h5py
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display, Markdown

from pim.world_models.loader import load_checkpoint, load_dataset
from pim.world_models.latent_dit import LatentDiTModel, LatentDiTState  # noqa: F401
from pim.extractors import fit_readability_probes
from pim.figures.theme import style_ax
from editability_metrics import build_edit_zones, edit_scorecard, fidelity_ratio

torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_OBJ, N_EDIT, K, N_PROBE = 2, 64, 15, 500
STEER_STEPS, STEER_LR, LAM_MAIN = 300, 0.02, 0.1
OUT = "/tmp/input_grad_steering_latent_dit"; os.makedirs(OUT, exist_ok=True)

MODEL_LABEL = "Latent DiT · z=16 · window 4"
model, info = load_checkpoint("../../../../runs/latent_dit/0_latent_dit_z16_w4/best_model.pt", device=DEVICE)
core = model.core
W, Z, D = model.window, model.latent_dim, core.cfg.d_model
S = core.cfg.n_sample_steps
taus_full = torch.linspace(1.0, 0.0, S + 1, device=DEVICE)

bundle = load_dataset("../../../../datasets/4_fixed_refl_inview", n_obj_keep=N_OBJ)
test, edits = bundle.test, bundle.edits
ef, R = edits.edit_frame, edits.obs_res
sim = test.config["dataset"]["sim"]
print(f"model {MODEL_LABEL} | epoch {info.epoch} decoded val MSE {info.val_loss:.5f} | "
      f"z={Z} window={W} d_model={D} | ef={ef} R={R} | N_EDIT={N_EDIT} K={K} device={DEVICE}")

def activations(state):
    """Differentiable activations-view flat state of the core (d_model)."""
    prev = core.state_view
    core.state_view = "activations"
    z = core.flat_state(core_state(state))
    core.state_view = prev
    return z

def core_state(state):
    from pim.world_models.dit.model import DiTState
    return DiTState(state.latent_buffer, state.length)

# probes on the activations state (τ=1 fixed-noise view, the model's default)
obs_probe = torch.from_numpy(test.obs[:N_PROBE]).float().to(DEVICE)
Tm1 = test.obs.shape[1] - 1
P_tgt = test.positions[:N_PROBE, :Tm1, :N_OBJ].reshape(N_PROBE, Tm1, N_OBJ * 2).astype(np.float32)
vis = test.is_visible[:N_PROBE, :Tm1, :N_OBJ].all(axis=2)
model.state_view = "activations"
with torch.no_grad():
    acts = model.get_hidden_states(obs_probe).cpu().numpy()
model.state_view = "latent_window"
PROBE = fit_readability_probes(acts, P_tgt, mask=vis, device=DEVICE)
for p in PROBE["mlp"].parameters():
    p.requires_grad_(False)
A_t = torch.from_numpy(PROBE["A"]).to(DEVICE); b_t = torch.from_numpy(PROBE["b"]).to(DEVICE)
display(Markdown(f"**Held-out position R² on the activations state**: linear **{PROBE['linear_r2']:.3f}**, "
                 f"MLP **{PROBE['mlp_r2']:.3f}** ({PROBE['n_train_seq']}/{PROBE['n_heldout_seq']} sequences)"))

In [ ]:
# [2] §4 machinery: edit zones, states, and both oracle references (single-frame and velocity-consistent window).
from pim.simulator.renderer import render_frame
from pim.simulator.sim import SimConfig

N = min(N_EDIT, edits.n_samples); ar = np.arange(N)
oe = edits.edit_object[:N].astype(int)
with h5py.File(edits.h5_path, "r") as f:
    pre_vel = f["velocities"][:N, ef - 1, :N_OBJ, :].astype(np.float32)
pre_pos = edits.positions[:N, ef - 1, :N_OBJ, :].astype(np.float32)
tgt_pos = edits.positions[:N, ef, :N_OBJ, :].astype(np.float32)
target4 = torch.from_numpy(tgt_pos.reshape(N, N_OBJ * 2)).float().to(DEVICE)
gt_roll = edits.clean_obs[:N, ef:ef + K, :].astype(np.float32)

ZONES = build_edit_zones(pre_pos=pre_pos, tgt_pos=tgt_pos, pre_vel=pre_vel, edit_object=oe, sim=sim,
                         n_obj=N_OBJ, traj_pos=edits.positions[:N, ef:ef + K, :N_OBJ, :].astype(np.float32),
                         gt_edited_traj=gt_roll)
teleport = ZONES.teleport
gt_edited_t = torch.from_numpy(ZONES.gt_edited).float().to(DEVICE)

obs_e = torch.from_numpy(edits.obs[:N]).float().to(DEVICE)
hist_obs = obs_e[:, :ef]                                  # frames 0..ef-1 (noisy, as teacher-forced)
base_frame = hist_obs[:, -1]
with torch.no_grad():
    hist_lat = model.encode(hist_obs)                     # (N, ef, Z) normalised latents
    base_lat = hist_lat[:, -1]
    gt_edited_lat = model.encode(gt_edited_t)
full_len = torch.full((N,), W, dtype=torch.long, device=DEVICE)

# Δ_true measured in BOTH spaces (each editor is scored in the space it writes in)
delta_true_obs = ZONES.gt_edited - base_frame.cpu().numpy()
delta_true_lat = (gt_edited_lat - base_lat).cpu().numpy()

def state_from_lat(latents):
    return LatentDiTState(latents[:, -W:].contiguous(), full_len)

@torch.no_grad()
def rollout_from(state, k=K):
    """Mean-mode free-run; step 0 decodes frame ef. Feedback stays in latent space."""
    model.predict_mode = "mean"
    z = core.decode(core_state(state)); s = state; out = []
    for _ in range(k):
        out.append(model.decode_latent(z).cpu().numpy())
        z, s = model.step_latent(z, s)
    return np.stack(out, axis=1)

CARDS, ROLLS, ARMS = {}, {}, {}
state_unsteered = state_from_lat(hist_lat)
ROLLS["Unsteered"] = rollout_from(state_unsteered)
CARDS["Unsteered"] = edit_scorecard(ROLLS["Unsteered"], ZONES, gt_roll)

# oracle 1: single-frame render write (newest latent ← encode(clean edited render))
lat_o1 = torch.cat([hist_lat[:, :-1], gt_edited_lat.unsqueeze(1)], dim=1)
ROLLS["Render write @1 (oracle)"] = rollout_from(state_from_lat(lat_o1))
CARDS["Render write @1 (oracle)"] = edit_scorecard(ROLLS["Render write @1 (oracle)"], ZONES, gt_roll)

# oracle 2: velocity-consistent counterfactual window write (all W frames)
dt = float(sim["dt"])
cfg_r = SimConfig(seed=0, y_near=sim["y_near"], y_far=sim["y_far"], x_near=sim["x_near"], x_far=sim["x_far"],
                  n_objects=N_OBJ, radius=sim["radius"], n_frames=1, dt=dt, obs_res=sim["obs_res"],
                  refl_min=sim["refl_min"], refl_max=sim["refl_max"], fixed_reflectivities=True,
                  obs_noise_std=0.0, boundary="open", always_in_frustum=False,
                  soft_edge=sim.get("soft_edge", 0.0), soft_shading=sim.get("soft_shading", "flat"),
                  soft_psf_sigma=sim.get("soft_psf_sigma", 0.0),
                  soft_occlusion_temp=sim.get("soft_occlusion_temp", 0.0))
refl = np.linspace(sim["refl_min"], sim["refl_max"], N_OBJ).astype(np.float32)
rad = np.full(N_OBJ, sim["radius"], np.float32)
offset = tgt_pos[ar, oe] - (pre_pos[ar, oe] + pre_vel[ar, oe] * dt)
cf_frames = np.zeros((N, W, R), np.float32)
for i in range(N):
    for j, t in enumerate(range(ef - W, ef)):
        pos_t = edits.positions[i, t, :N_OBJ].astype(np.float32).copy()
        pos_t[oe[i]] += offset[i]
        _, _, inten = render_frame(pos_t, rad, refl, cfg_r)
        cf_frames[i, j] = inten
land = pre_pos[ar, oe] + offset + pre_vel[ar, oe] * dt
print(f"counterfactual construction check: max |landing − target| = "
      f"{np.abs(land - tgt_pos[ar, oe]).max():.2e} sim-units")
with torch.no_grad():
    cf_lat = model.encode(torch.from_numpy(cf_frames).float().to(DEVICE))
ROLLS["Counterfactual window write (n=4, oracle)"] = rollout_from(state_from_lat(cf_lat))
CARDS["Counterfactual window write (n=4, oracle)"] = edit_scorecard(
    ROLLS["Counterfactual window write (n=4, oracle)"], ZONES, gt_roll)

print(f"N={N} edits | mean teleport {teleport.mean():.2f} sim-units")
for k_ in ["Unsteered", "Render write @1 (oracle)", "Counterfactual window write (n=4, oracle)"]:
    print(f"  {k_:44s} Edit Index {CARDS[k_]['edit_index']:+.2f}")

In [ ]:
# [3] Editors A & B — gradient steering on the two clean write surfaces (observation vs latent window).
def _readout(state, probe):
    z = activations(state)
    return z @ A_t.T + b_t if probe == "linear" else PROBE["mlp"](z)

def obs_grad_steer(probe="linear", n=1, lam=LAM_MAIN, steps=STEER_STEPS, lr=STEER_LR):
    """Adam on δ over the newest n OBSERVATION frames; backprop through the VAE encoder + core."""
    delta = torch.zeros(N, n, R, device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam([delta], lr=lr)
    first = None
    for it in range(steps):
        frames = torch.cat([hist_obs[:, :ef - n], (hist_obs[:, ef - n:] + delta).clamp(0, 1)], dim=1)
        st = state_from_lat(model.encode(frames))
        loss = ((_readout(st, probe) - target4) ** 2).sum(-1).mean() + lam * (delta ** 2).sum((-1, -2)).mean()
        opt.zero_grad(); loss.backward()
        if it == 0:
            first = -delta.grad[:, -1].detach().clone()
        opt.step()
    with torch.no_grad():
        frames = torch.cat([hist_obs[:, :ef - n], (hist_obs[:, ef - n:] + delta).clamp(0, 1)], dim=1)
        st = state_from_lat(model.encode(frames))
        after = (_readout(st, probe) - target4).norm(dim=-1).mean().item()
        before = (_readout(state_unsteered, probe) - target4).norm(dim=-1).mean().item()
        d_eff = (frames[:, -1] - base_frame).cpu().numpy()
    return dict(state=st, delta=d_eff, first_grad=first.cpu().numpy(), space="obs",
                resid_before=before, resid_after=after)

def latent_grad_steer(probe="linear", n=1, lam=LAM_MAIN, steps=STEER_STEPS, lr=STEER_LR):
    """Adam on δ over the newest n LATENTS of the carried window (the compressed surface)."""
    delta = torch.zeros(N, n, Z, device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam([delta], lr=lr)
    first = None
    for it in range(steps):
        lat = torch.cat([hist_lat[:, ef - W:ef - n], hist_lat[:, ef - n:] + delta], dim=1)
        st = LatentDiTState(lat, full_len)
        loss = ((_readout(st, probe) - target4) ** 2).sum(-1).mean() + lam * (delta ** 2).sum((-1, -2)).mean()
        opt.zero_grad(); loss.backward()
        if it == 0:
            first = -delta.grad[:, -1].detach().clone()
        opt.step()
    with torch.no_grad():
        lat = torch.cat([hist_lat[:, ef - W:ef - n], hist_lat[:, ef - n:] + delta], dim=1)
        st = LatentDiTState(lat, full_len)
        after = (_readout(st, probe) - target4).norm(dim=-1).mean().item()
        before = (_readout(state_unsteered, probe) - target4).norm(dim=-1).mean().item()
        d_eff = (lat[:, -1] - base_lat).cpu().numpy()
    return dict(state=st, delta=d_eff, first_grad=first.cpu().numpy(), space="latent",
                resid_before=before, resid_after=after)

SPECS = []
for probe in ("linear", "MLP"):
    p = probe.lower() if probe == "linear" else "mlp"
    SPECS += [(f"Obs Grad · {probe} probe (n=1, λ={LAM_MAIN})", obs_grad_steer, dict(probe=p, n=1)),
              (f"Obs Grad · {probe} probe (n={W} whole window, λ={LAM_MAIN})", obs_grad_steer, dict(probe=p, n=W)),
              (f"Latent-window Grad · {probe} probe (n=1, λ={LAM_MAIN})", latent_grad_steer, dict(probe=p, n=1)),
              (f"Latent-window Grad · {probe} probe (n={W} whole window, λ={LAM_MAIN})", latent_grad_steer, dict(probe=p, n=W))]
for name, fn, kw in SPECS:
    a = fn(**kw)
    ARMS[name] = a
    ROLLS[name] = rollout_from(a["state"])
    CARDS[name] = edit_scorecard(ROLLS[name], ZONES, gt_roll)
    print(f"{name:56s} probe residual {a['resid_before']:.3f} → {a['resid_after']:.3f} | "
          f"Edit Index {CARDS[name]['edit_index']:+.2f}")

In [ ]:
# [4] Editor C — Iterate Grad Steering @(Lℓ, τ_pause): pause–optimize–resume on the diffusion iterate,
#     with probes fit AND held-out-verified at the same (residual point, τ) on Euler-iterate states.
L_STEER = core.cfg.n_layers - 1          # "late" residual point (the pixel-DiT grid's best); see world-state nb
PAUSE_KS = [2, 4, 6]                     # τ_pause = 0.75, 0.50, 0.25

with torch.no_grad():
    z_probe_seq = model.encode(obs_probe)
win_p, len_p = core._unfold_windows(z_probe_seq)
flat_wp = win_p.reshape(-1, W, Z)
flat_lp = len_p.unsqueeze(0).expand(N_PROBE, -1).reshape(-1)
n_rows = flat_wp.shape[0]
lat_states = {k: np.empty((n_rows, D), np.float32) for k in PAUSE_KS}
genp = torch.Generator().manual_seed(7)
eps_rows = torch.randn(n_rows, Z, generator=genp)
with torch.no_grad():
    for i0 in range(0, n_rows, 8192):
        sl = slice(i0, min(i0 + 8192, n_rows))
        cur, nxt = core._window_tokens(flat_wp[sl])
        attn = core._window_attn_mask(flat_lp[sl], DEVICE)
        n = cur.shape[0]
        x = eps_rows[sl].to(DEVICE)
        for k in range(S):
            tau_t = torch.zeros(n, W, device=DEVICE); tau_t[:, -1] = taus_full[k]
            nxt_k = torch.cat([nxt[:, :-1], x.unsqueeze(1)], dim=1)
            if k in PAUSE_KS:
                sink = []
                feats, c = core._trunk(cur, nxt_k, tau_t, attn, resid_sink=sink)
                lat_states[k][sl] = sink[L_STEER][:, -1].cpu().numpy()
                v = core.final_layer(feats, c)[:, -1]
            else:
                v = core._denoise(cur, nxt_k, tau_t, attn)[:, -1]
            x = x + (taus_full[k + 1] - taus_full[k]) * v

P_p = test.positions[:N_PROBE, :Tm1, :N_OBJ].reshape(N_PROBE, Tm1, N_OBJ * 2).astype(np.float32)
ITER_PROBES = {}
for k in PAUSE_KS:
    fit = fit_readability_probes(lat_states[k].reshape(N_PROBE, Tm1, D), P_p, mask=vis, device=DEVICE)
    for p in fit["mlp"].parameters():
        p.requires_grad_(False)
    ITER_PROBES[k] = fit
    print(f"steering probe @ (L={L_STEER}, τ={float(taus_full[k]):.2f}): held-out R² linear "
          f"{fit['linear_r2']:.3f} | MLP {fit['mlp_r2']:.3f} — verified at the steering point")

def iterate_grad_steer(k_pause, probe="linear", steps=STEER_STEPS, lr=0.05, seed=456):
    fit = ITER_PROBES[k_pause]
    A = torch.from_numpy(fit["A"]).to(DEVICE); b = torch.from_numpy(fit["b"]).to(DEVICE)
    buf = state_unsteered.latent_buffer
    cur, nxt = core._window_tokens(buf)
    attn = core._window_attn_mask(full_len, DEVICE)
    gen = torch.Generator().manual_seed(seed)
    x = torch.randn(N, Z, generator=gen).to(DEVICE)

    def v_at(xc, k):
        tau_t = torch.zeros(N, W, device=DEVICE); tau_t[:, -1] = taus_full[k]
        return core._denoise(cur, torch.cat([nxt[:, :-1], xc.unsqueeze(1)], 1), tau_t, attn)[:, -1]

    def read(xc):
        tau_t = torch.zeros(N, W, device=DEVICE); tau_t[:, -1] = taus_full[k_pause]
        sink = []
        core._trunk(cur, torch.cat([nxt[:, :-1], xc.unsqueeze(1)], 1), tau_t, attn, resid_sink=sink)
        h = sink[L_STEER][:, -1]
        return h @ A.T + b if probe == "linear" else fit["mlp"](h)

    with torch.no_grad():
        for k in range(k_pause):
            x = x + (taus_full[k + 1] - taus_full[k]) * v_at(x, k)
        before = (read(x) - target4).norm(dim=-1).mean().item()
    delta = torch.zeros_like(x, requires_grad=True)
    opt = torch.optim.Adam([delta], lr=lr)
    for _ in range(steps):
        loss = ((read(x + delta) - target4) ** 2).sum(-1).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        x = (x + delta).detach()
        after = (read(x) - target4).norm(dim=-1).mean().item()
        dn = float(delta.norm(dim=-1).mean())
        for k in range(k_pause, S):
            x = x + (taus_full[k + 1] - taus_full[k]) * v_at(x, k)
    return x, before, after, dn

@torch.no_grad()
def rollout_from_latent(z0, k=K):
    """Rollout whose step-0 latent is supplied by an editor (then mean-mode free-run)."""
    model.predict_mode = "mean"
    out, s, z = [], state_unsteered, z0
    for _ in range(k):
        out.append(model.decode_latent(z).cpu().numpy())
        z, s = model.step_latent(z, s)
    return np.stack(out, axis=1)

ITER_LAT = {}
for probe in ("linear", "MLP"):
    p = "linear" if probe == "linear" else "mlp"
    for k_ in PAUSE_KS:
        name = f"Iterate Grad · {probe} probe @(L{L_STEER}, τ={float(taus_full[k_]):.2f})"
        z0, r0, r1, dn = iterate_grad_steer(k_, probe=p)
        ITER_LAT[name] = z0
        ROLLS[name] = rollout_from_latent(z0)
        CARDS[name] = edit_scorecard(ROLLS[name], ZONES, gt_roll)
        print(f"{name:52s} residual {r0:.3f} → {r1:.3f} | ‖δ‖(latent) {dn:.3f} | "
              f"Edit Index {CARDS[name]['edit_index']:+.2f} | collateral {CARDS[name]['collateral_rmse']:.3f}")

In [ ]:
# [5] Fig 1 — the headline diagnostic: is the gradient aligned with the true edit direction, in the space it
#     writes in? Each editor is compared against Δ_true measured in ITS OWN space (obs-space vs latent-space),
#     each with its own empirical shuffled-pair chance level.
def cos_stats(vecs, dtrue):
    num = (vecs * dtrue).sum(-1)
    den = np.linalg.norm(vecs, axis=-1) * np.linalg.norm(dtrue, axis=-1) + 1e-12
    c = num / den
    return float(c.mean()), float(np.degrees(np.arccos(np.clip(c, -1, 1))).mean())

chance = {}
for space, dtrue in (("obs", delta_true_obs), ("latent", delta_true_lat)):
    ref = next(a for a in ARMS.values() if a["space"] == space)
    chance[space] = cos_stats(np.roll(ref["delta"], 1, axis=0), dtrue)[0]

rows, plot = [], []
for name, a in ARMS.items():
    dtrue = delta_true_obs if a["space"] == "obs" else delta_true_lat
    c_o, a_o = cos_stats(a["delta"], dtrue)
    c_g, a_g = cos_stats(a["first_grad"], dtrue)
    rows.append(f"| {name} | {a['space']} | {c_o:+.3f} | {a_o:.0f}° | {c_g:+.3f} | {chance[a['space']]:+.3f} |")
    plot.append((name, a["space"], c_o, c_g))
display(Markdown("**Alignment of the steering perturbation with the true edit direction** — each editor scored "
                 "in the space it writes in, against that space's own shuffled-pair chance level. "
                 "**Pixel-space reference (cited, `input_grad_steering_dit.ipynb`): cos ≈ +0.11…+0.15 linear, "
                 "+0.22 MLP.**\n\n| editor | write space | cos(δ*, Δ_true) | angle | cos(first grad, Δ_true) "
                 "| shuffled-pair chance |\n|---|---|---|---|---|---|\n" + "\n".join(rows)))

fig, ax = plt.subplots(figsize=(9, 0.42 * len(plot) + 1.6))
y = np.arange(len(plot))
colors = ["#0072B2" if s == "latent" else "#E69F00" for _, s, _, _ in plot]
ax.barh(y, [c for _, _, c, _ in plot], color=colors, height=0.62)
for space, col, ls in (("obs", "#E69F00", "--"), ("latent", "#0072B2", ":")):
    ax.axvline(chance[space], color=col, ls=ls, lw=1.3, label=f"{space}-space shuffled-pair chance")
ax.axvline(0.15, color="#666666", lw=1.2, ls="-.", label="pixel-DiT observation-space reference (≈0.15)")
ax.set_yticks(y); ax.set_yticklabels([n for n, _, _, _ in plot], fontsize=8); ax.invert_yaxis()
ax.set_xlabel("cosine of the converged perturbation with Δ_true (in its own write space)")
ax.set_title(f"Fig 1 — does the probe gradient point at the real edit? ({MODEL_LABEL})", fontsize=11)
handles = [Line2D([0], [0], color="#0072B2", lw=6, label="latent-space write"),
           Line2D([0], [0], color="#E69F00", lw=6, label="observation-space write")] + ax.get_legend_handles_labels()[0]
ax.legend(handles=handles, fontsize=7.5, loc="lower right")
style_ax(ax); fig.tight_layout()
fig.savefig(f"{OUT}/fig1_cosine.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# [6] §4 scorecard (all arms) + Fig 2, the cross-architecture ladder: latent DiT vs pixel DiT, same editors.
ORDER = (["Unsteered"] + [n for n, _, _ in SPECS]
         + [n for n in CARDS if n.startswith("Iterate Grad")]
         + ["Render write @1 (oracle)", "Counterfactual window write (n=4, oracle)"])
hdr = ("| arm | Edit Index (−1…+1) | edit-frame RMSE | target RMSE | ghost RMSE | collateral RMSE "
       "| GT-traj RMSE | fidelity ratio |\n|---|---|---|---|---|---|---|---|\n")
rows = []
for name in ORDER:
    c = CARDS[name]
    rows.append(f"| {name} | {c['edit_index']:+.2f} | {c['edit_frame_rmse']:.3f} | {c['target_rmse']:.3f} "
                f"| {c['ghost_rmse']:.3f} | {c['collateral_rmse']:.3f} | {c['gt_traj_rmse']:.3f} "
                f"| {fidelity_ratio(c, CARDS['Unsteered']):.2f} |")
display(Markdown("**§4 scorecard — every arm** (RMSE in intensity units vs the clean edited-world render; step 0 "
                 "= frame ef). Watch the collateral column: an index that moved while collateral exploded is "
                 "degradation, not editing.\n\n" + hdr + "\n".join(rows)))

# Fig 2 — matched-editor comparison against the pixel DiT (values cited from input_grad_steering_dit.ipynb)
PIXEL = {"Unsteered": -0.65,
         "history/obs grad · linear (n=1)": -0.52,
         "history/obs grad · linear (whole window)": -0.52,
         "history/obs grad · MLP (n=1)": -0.31,
         "history/obs grad · MLP (whole window)": -0.31,
         "iterate grad · linear (best τ)": -0.18,
         "iterate grad · MLP (best τ)": -0.21,
         "Render write @1 (oracle)": +0.12,
         "Counterfactual window write (oracle)": +0.71}
def best(prefix):
    vals = [CARDS[n]["edit_index"] for n in CARDS if n.startswith(prefix)]
    return max(vals) if vals else np.nan
LATENT = {"Unsteered": CARDS["Unsteered"]["edit_index"],
          "history/obs grad · linear (n=1)": CARDS[f"Obs Grad · linear probe (n=1, λ={LAM_MAIN})"]["edit_index"],
          "history/obs grad · linear (whole window)": CARDS[f"Obs Grad · linear probe (n={W} whole window, λ={LAM_MAIN})"]["edit_index"],
          "history/obs grad · MLP (n=1)": CARDS[f"Obs Grad · MLP probe (n=1, λ={LAM_MAIN})"]["edit_index"],
          "history/obs grad · MLP (whole window)": CARDS[f"Obs Grad · MLP probe (n={W} whole window, λ={LAM_MAIN})"]["edit_index"],
          "iterate grad · linear (best τ)": best("Iterate Grad · linear"),
          "iterate grad · MLP (best τ)": best("Iterate Grad · MLP"),
          "Render write @1 (oracle)": CARDS["Render write @1 (oracle)"]["edit_index"],
          "Counterfactual window write (oracle)": CARDS["Counterfactual window write (n=4, oracle)"]["edit_index"]}
EXTRA = {"latent-window grad · linear (n=1)": CARDS[f"Latent-window Grad · linear probe (n=1, λ={LAM_MAIN})"]["edit_index"],
         "latent-window grad · linear (whole window)": CARDS[f"Latent-window Grad · linear probe (n={W} whole window, λ={LAM_MAIN})"]["edit_index"],
         "latent-window grad · MLP (n=1)": CARDS[f"Latent-window Grad · MLP probe (n=1, λ={LAM_MAIN})"]["edit_index"],
         "latent-window grad · MLP (whole window)": CARDS[f"Latent-window Grad · MLP probe (n={W} whole window, λ={LAM_MAIN})"]["edit_index"]}

labels = list(PIXEL.keys())
y = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(9.5, 0.46 * (len(labels) + len(EXTRA)) + 1.8))
ax.barh(y - 0.2, [PIXEL[k] for k in labels], height=0.4, color="#999999", label="pixel DiT · d256 · window 4 (cited)")
ax.barh(y + 0.2, [LATENT[k] for k in labels], height=0.4, color="#0072B2", label=MODEL_LABEL)
y2 = np.arange(len(labels), len(labels) + len(EXTRA))
ax.barh(y2 + 0.2, list(EXTRA.values()), height=0.4, color="#56B4E9",
        label="latent-DiT only: no pixel-space counterpart exists")
ax.axvline(0, color="#555555", lw=0.9)
ax.set_yticks(np.concatenate([y, y2]))
ax.set_yticklabels(labels + list(EXTRA.keys()), fontsize=8.5)
ax.invert_yaxis(); ax.set_xlim(-1.02, 1.02)
ax.set_xlabel("Edit Index at the edit frame (−1 = unedited world, +1 = edited world, ≈0 = equidistant/garbage)")
ax.set_title("Fig 2 — same editors, two architectures: does a 16-d semantic bottleneck make probe gradients "
             "controllable?", fontsize=11)
ax.legend(fontsize=8, loc="lower right"); style_ax(ax); fig.tight_layout()
fig.savefig(f"{OUT}/fig2_ladder.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# [7] Fig 3 — observation-space waterfall (canonical fixed spec; one helper, every waterfall through it).
N_CTX = 6
ctx_obs = edits.obs[:N, ef - N_CTX:ef, :].astype(np.float32)
DARK, TXT, TICK, EDIT_C = "#0a0a14", "#a3adc2", "#808a9d", "#fa8850"
TARGET_C, GHOST_C = "#00E676", "#FF5252"
def _cx(m):
    i = np.where(m)[0]; return i.mean() if i.size else np.nan
tgt_cx = np.array([_cx(ZONES.target[i]) for i in range(N)])
pre_cx = np.array([_cx(ZONES.ghost[i]) for i in range(N)])
SAMPLES = list(np.argsort(teleport * (ZONES.ghost.sum(1) >= 3))[::-1][:3])

def waterfall_grid(col_titles, col_bodies, samples, suptitle, fname):
    """col_bodies[c]: (N, L, R) rows BELOW the N_CTX context frames; every column its OWN free-run
    from step 0 (= frame ef). No shared teacher-forced ef row (banned; see CLAUDE.md)."""
    ncol = len(col_titles)
    fig, axes = plt.subplots(len(samples), ncol, figsize=(3.25 * ncol, 3.4 * len(samples)),
                             squeeze=False, facecolor=DARK)
    for r, smp in enumerate(samples):
        for c in range(ncol):
            ax = axes[r][c]; ax.set_facecolor(DARK)
            panel = np.clip(np.concatenate([ctx_obs[smp], col_bodies[c][smp]], axis=0), 0, 1)
            ax.imshow(panel, aspect="auto", origin="upper", cmap="gray", vmin=0, vmax=1,
                      interpolation="nearest")
            for sp in ax.spines.values(): sp.set_edgecolor(TICK)
            ax.axhline(N_CTX - 0.5, color=EDIT_C, lw=1.4, ls="--", alpha=0.95)
            if not np.isnan(tgt_cx[smp]): ax.axvline(tgt_cx[smp], color=TARGET_C, lw=1.6, alpha=0.9)
            if not np.isnan(pre_cx[smp]): ax.axvline(pre_cx[smp], color=GHOST_C, ls="--", lw=1.6, alpha=0.9)
            if r == 0: ax.set_title(col_titles[c], fontsize=7.5, color=TXT)
            if c == 0:
                ax.set_ylabel(f"sample {smp} (teleport {teleport[smp]:.1f})\nsim frame", fontsize=8, color=TXT)
                ax.set_yticks([0, N_CTX, N_CTX + 7, N_CTX + 14])
                ax.set_yticklabels([ef - N_CTX, ef, ef + 7, ef + 14], fontsize=7)
            else: ax.set_yticks([])
            ax.set_xlabel("ray", fontsize=8, color=TXT); ax.tick_params(colors=TICK, labelsize=7)
    handles = [Line2D([0], [0], color=TARGET_C, lw=2.2, label="object target location"),
               Line2D([0], [0], color=GHOST_C, ls="--", lw=2.2, label="ghost (pre-edit) location"),
               Line2D([0], [0], color=EDIT_C, ls="--", lw=2.2,
                      label=f"edit applied here ({N_CTX} noisy context frames above; every row below is that "
                            f"column's OWN free-run, step 0 = frame {ef})")]
    fig.legend(handles=handles, loc="upper center", ncol=2, fontsize=8.5, frameon=False,
               labelcolor=TXT, bbox_to_anchor=(0.5, 0.965))
    fig.suptitle(suptitle, y=1.0, fontsize=10.5, color=TXT)
    fig.tight_layout(rect=[0, 0, 1, 0.90])
    fig.savefig(fname, dpi=150, facecolor=DARK, bbox_inches="tight"); plt.show()

best_iter = max((n for n in CARDS if n.startswith("Iterate Grad")), key=lambda n: CARDS[n]["edit_index"])
WF = ["Unsteered",
      f"Obs Grad · MLP probe (n=1, λ={LAM_MAIN})",
      f"Latent-window Grad · linear probe (n=1, λ={LAM_MAIN})",
      f"Latent-window Grad · MLP probe (n={W} whole window, λ={LAM_MAIN})",
      best_iter, "Render write @1 (oracle)", "Counterfactual window write (n=4, oracle)"]
import textwrap
titles = ["GT (sim clean obs)"] + ["\n".join(textwrap.wrap(n, 30)) +
                                   f"\nEdit Index {CARDS[n]['edit_index']:+.2f}" for n in WF]
waterfall_grid(titles, [gt_roll] + [ROLLS[n] for n in WF], SAMPLES,
               f"Fig 3 — free-run waterfalls: gradient steering on both write surfaces vs oracles ({MODEL_LABEL})",
               f"{OUT}/fig3_waterfall.png")

## Current results (updated 2026-08-11)

**The headline: compressing the write surface 128-d → 16-d does NOT make probe gradients semantic.**

- **cos(δ*, Δ_true) is unchanged by compression.** Latent-space writes: **+0.118 … +0.168** (80–83°).
  Observation-space writes on the same model: +0.146 … +0.212. Pixel DiT (cited): +0.11 … +0.22. Every
  shuffled-pair chance level ≈ −0.03. An 8× semantic bottleneck, where nearly every direction should be
  meaningful, produces gradients no better aligned with the true edit than 128-d pixel space did.
- **Edit Index band, all probe-only editors: −0.13 … −0.48**, matching the pixel DiT arm-for-arm (Fig 2):
  obs-grad linear −0.37/−0.44 (pixel −0.52), obs-grad MLP −0.30/−0.33 (pixel −0.31), latent-window grad
  −0.38 … −0.48 (**no pixel counterpart** — the compressed surface is not better than the raw one),
  iterate grad −0.13 … −0.23 (pixel −0.18/−0.21). Probe capacity and window width move things by ≲0.1, as before.
- **Both oracles land exactly where they did in pixel space**: Render write @1 **+0.12** (identical), and the
  velocity-consistent **Counterfactual window write +0.71** (identical), with collateral at the unsteered
  baseline and GT-traj RMSE 0.200 *better* than unsteered 0.303. The architecture is fully editable through
  consistent multi-frame evidence; only the gradient-found content fails.
- **More readable, not more controllable.** This model's state is the most position-readable in the thread —
  linear R² **0.810** / MLP **0.905** on the activations state (pixel DiT: 0.70 / 0.85), and 0.79–0.81 linear at
  every (L3, τ) steering point — yet its editability is unchanged. The clearest readable≠controllable
  demonstration we have.
- Iterate-grad arms again show the duplication signature (collateral 0.34–0.40 vs unsteered 0.147) rather than
  relocation, and remain flat across τ_pause.

## Summary (interpretation — clearly marked as such)

The "dimensionality of adversarial freedom" hypothesis is **refuted for this world**: the probe-gradient failure
survives an 8× semantic compression intact, on a model whose latent is *more* linearly readable than the pixel
model's state and whose decoder is an additional unconditional projector back to valid observations. Since
representation geometry does not explain it, what remains is **belief dynamics**: the ghost is carried by the
clean context frames, and only evidence that is consistent *across the window* (the +0.71 counterfactual write)
removes it — a gradient that edits one frame, or one iterate, can add mass at the target but cannot retract the
context's testimony about where the object was. The practical corollary for this thread: stop looking for a
better *space* to take the gradient in, and start looking for objectives that produce multi-frame,
velocity-consistent evidence (SDEdit-style re-noise/re-denoise of the whole window; a render-space objective;
or repeated small edits across several generated frames).